In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model and tokenizer
model_name = "microsoft/DialoGPT-medium"

print("Loading model... Please wait.")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Store conversation history
chat_history_ids = None

print("\n====================================")
print(" Chatbot Ready! Type 'quit' to exit.")
print("====================================\n")

while True:

    # Get user input
    user_input = input("You: ")

    if user_input.lower() == "quit":
        print("Bot: Goodbye! Have a nice day.")
        break

    # Encode user input
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    # Append previous chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Attention mask
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long).to(device)

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_new_tokens=60,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.2
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Bot:", response)

Loading model... Please wait.


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


 Chatbot Ready! Type 'quit' to exit.

You: how are you?
Bot: Good , how about yourself ?
You: what is Machine Learning
Bot: What's that , a machine learning algorithm for the purpose of teaching people to use it ?
You: quit
Bot: Goodbye! Have a nice day.
